In [ ]:
import pandas as pd
import sqlite3

# Load your data (upload sales_data_sample.csv first using the folder icon on the left)
df = pd.read_csv("sales_data_sample.csv", encoding="latin1")

# Create a SQLite database and load the data into a table
conn = sqlite3.connect("sales.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

print("Database created with", len(df), "rows")


Database created with 2823 rows


In [ ]:
%load_ext sql
%sql sqlite:///sales.db

In [ ]:
query = """
SELECT *
FROM sales
WHERE COUNTRY = 'Germany';
"""
print(pd.read_sql(query, conn))

    ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0         10191               21     100.00                3  3840.90   
1         10300               33     100.00                5  5521.89   
2         10310               33     100.00               10  6934.62   
3         10230               43     100.00                1  7016.31   
4         10191               40     100.00                1  5590.00   
..          ...              ...        ...              ...      ...   
57        10296               34     100.00               11  3477.86   
58        10296               24     100.00                4  2441.04   
59        10296               22      80.80                3  1777.60   
60        10296               47      86.62                5  4071.14   
61        10296               21      45.19               10   948.99   

          ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   11/20/2003 0:00  Shipped       4        11     2003  ..

In [ ]:
query = """
SELECT *
FROM sales
WHERE COUNTRY = 'Germany' AND SALES > 3000
ORDER BY SALES DESC;
"""
print(pd.read_sql(query, conn))

    ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0         10310               48     100.00                3  8940.96   
1         10230               49     100.00                8  7300.51   
2         10230               42     100.00                3  7238.28   
3         10230               43     100.00                1  7016.31   
4         10310               33     100.00               10  6934.62   
5         10310               49     100.00               12  6266.12   
6         10310               37     100.00                2  6231.91   
7         10323               47     100.00                1  6203.06   
8         10296               36     100.00                7  5676.84   
9         10191               40     100.00                1  5590.00   
10        10300               33     100.00                5  5521.89   
11        10310               45     100.00                5  5497.65   
12        10310               40     100.00        

In [ ]:
query = """
SELECT YEAR_ID, PRODUCTLINE, ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
GROUP BY YEAR_ID, PRODUCTLINE
ORDER BY YEAR_ID, total_revenue DESC;
"""
print(pd.read_sql(query, conn))

    YEAR_ID       PRODUCTLINE  total_revenue
0      2003      Classic Cars     1484785.29
1      2003      Vintage Cars      650987.76
2      2003  Trucks and Buses      420429.93
3      2003       Motorcycles      370895.58
4      2003            Planes      272257.60
5      2003             Ships      244821.09
6      2003            Trains       72802.29
7      2004      Classic Cars     1762257.09
8      2004      Vintage Cars      911423.77
9      2004       Motorcycles      560545.23
10     2004  Trucks and Buses      529302.89
11     2004            Planes      502671.80
12     2004             Ships      341437.97
13     2004            Trains      116523.85
14     2005      Classic Cars      672573.28
15     2005      Vintage Cars      340739.31
16     2005       Motorcycles      234947.53
17     2005            Planes      200074.17
18     2005  Trucks and Buses      178057.02
19     2005             Ships      128178.07
20     2005            Trains       36917.33


In [ ]:
query = """
SELECT PRODUCTLINE, COUNT(DISTINCT CUSTOMERNAME) AS unique_customers
FROM sales
GROUP BY PRODUCTLINE
ORDER BY unique_customers DESC;
"""
print(pd.read_sql(query, conn))

        PRODUCTLINE  unique_customers
0      Classic Cars                89
1      Vintage Cars                84
2       Motorcycles                49
3  Trucks and Buses                48
4             Ships                48
5            Planes                47
6            Trains                34


In [ ]:
query="""
SELECT AVG(customer_total) FROM (
    SELECT CUSTOMERNAME, SUM(SALES) AS customer_total
    FROM sales
    GROUP BY CUSTOMERNAME
);
"""
print(pd.read_sql(query, conn))

   AVG(customer_total)
0        109050.313587


In [ ]:
query = """
SELECT CUSTOMERNAME, ROUND(SUM(SALES), 2) AS customer_total
FROM sales
GROUP BY CUSTOMERNAME
HAVING SUM(SALES) > (
    SELECT AVG(customer_total) FROM (
        SELECT CUSTOMERNAME, SUM(SALES) AS customer_total
        FROM sales
        GROUP BY CUSTOMERNAME
    )
)
ORDER BY customer_total DESC;
"""
print(pd.read_sql(query, conn))

                    CUSTOMERNAME  customer_total
0          Euro Shopping Channel       912294.11
1   Mini Gifts Distributors Ltd.       654858.06
2     Australian Collectors, Co.       200995.41
3             Muscle Machine Inc       197736.94
4              La Rochelle Gifts       180124.90
5        Dragon Souveniers, Ltd.       172989.68
6              Land of Toys Inc.       164069.44
7      The Sharp Gifts Warehouse       160010.27
8                 AV Stores, Co.       157807.81
9        Anna's Decorations, Ltd       153996.13
10     Souveniers And Things Co.       151570.98
11      Corporate Gift Ideas Co.       149882.50
12         Salzburg Collectables       149798.63
13      Danish Wholesale Imports       145041.60
14        Saveley & Henriot, Co.       142874.25
15           L'ordine Souveniers       142601.33
16                 Rovelli Gifts       137955.72
17            Reims Collectables       135042.94
18       Scandinavian Gift Ideas       134259.33
19  Online Diecast C

In [ ]:
query = """
SELECT
    CASE
        WHEN STATUS = 'Shipped' THEN 'Completed'
        ELSE 'Not Completed'
    END AS order_status,
    COUNT(*) AS num_orders,
    ROUND(SUM(SALES), 2) AS total_revenue
FROM sales
GROUP BY order_status;
"""
print(pd.read_sql(query, conn))

    order_status  num_orders  total_revenue
0      Completed        2617     9291501.08
1  Not Completed         206      741127.77


In [ ]:
query = """
SELECT
    PRODUCTLINE,
    ROUND(SUM(SALES), 2) AS total_revenue,
    RANK() OVER (ORDER BY SUM(SALES) DESC) AS revenue_rank
FROM sales
GROUP BY PRODUCTLINE;
"""
print(pd.read_sql(query, conn))

        PRODUCTLINE  total_revenue  revenue_rank
0      Classic Cars     3919615.66             1
1      Vintage Cars     1903150.84             2
2       Motorcycles     1166388.34             3
3  Trucks and Buses     1127789.84             4
4            Planes      975003.57             5
5             Ships      714437.13             6
6            Trains      226243.47             7
